# Rugo — Space Missions Example

This notebook demonstrates reading, filtering, aggregating, and writing columnar data
using **rugo** — no PyArrow, no NumPy, no heavy dependencies.

Dataset: [Space Missions](https://storage.googleapis.com/opteryx/space_missions/space_missions.parquet) — a catalogue of orbital launches from 1957 to the present.

In [ ]:
!pip install rugo

In [ ]:
# Download the dataset (skip if already present)
import os
if not os.path.exists("space_missions.parquet"):
    !wget -q https://storage.googleapis.com/opteryx/space_missions/space_missions.parquet
print("Dataset ready:", os.path.getsize("space_missions.parquet"), "bytes")

## 1. Schema — list all column names

`read_metadata` parses only the Parquet footer — no column data is decoded.

In [ ]:
from rugo import parquet

meta = parquet.read_metadata("space_missions.parquet")

print(f"Rows : {meta.num_rows:,}")
print(f"Columns ({len(meta.schema_columns)}):")
for col in meta.schema_columns:
    nullable = "nullable" if col.nullable else "not null"
    print(f"  {col.name:<30}  {col.physical_type}  ({nullable})")

## 3. Filter — missions launched by SpaceX

`filters=` prunes whole row groups via footer statistics; matching rows are then
verified at the row level after decode.

In [ ]:
with parquet.read_parquet("space_missions.parquet") as reader:
    first_morsel = next(iter(reader))

# The morsel: shape, column count, row count
print(first_morsel)

# A single column vector
company_vector = first_morsel.column(b"Company")
print(company_vector)

# First few values as a Python list
print(company_vector.to_pylist()[:10])

## 4. Aggregate — total launch spend (Price) by company

Stream one morsel (row group) at a time and accumulate totals — no full table
in memory.

In [ ]:
COMPANY = "SpaceX"

def _str(v):
    return v.decode() if isinstance(v, (bytes, bytearray)) else v

spacex = []

with parquet.read_parquet(
    "space_missions.parquet",
    columns=["Company", "Location", "Price"],
    filters=[("Company", "=", COMPANY)],
) as reader:
    for morsel in reader:
        companies = morsel.column(b"Company").to_pylist()
        locations = morsel.column(b"Location").to_pylist()
        prices    = morsel.column(b"Price").to_pylist()
        for company, location, price in zip(companies, locations, prices):
            if _str(company) == COMPANY:
                spacex.append({"Company": _str(company), "Location": _str(location), "Price": price})

print(f"{COMPANY} missions: {len(spacex):,}")
for row in spacex[:5]:
    print(f"  {row['Location']:<50}  ${row['Price']}")

## 5. Write to JSONL using rugo and inspect the output

`write_jsonl(morsel)` serializes a Draken Morsel to JSONL bytes directly in C++ —
no Python json module involved.  Here we write every row group of the full dataset
to a single file, then use `head` to confirm the output.

In [ ]:
from collections import defaultdict

totals = defaultdict(float)
counts = defaultdict(int)

with parquet.read_parquet("space_missions.parquet", columns=["Company", "Price"]) as reader:
    for morsel in reader:
        for company, price in zip(
            morsel.column(b"Company").to_pylist(),
            morsel.column(b"Price").to_pylist(),
        ):
            company = _str(company)
            counts[company] += 1
            if price is not None:
                totals[company] += price

ranked = sorted(totals.items(), key=lambda kv: kv[1], reverse=True)[:10]
print(f"{'Company':<35}  {'Missions':>8}  {'Total Price ($M)':>16}")
print("-" * 64)
for company, total in ranked:
    print(f"{company:<35}  {counts[company]:>8,}  {total:>16,.1f}")

## 4. Write to JSONL using rugo and inspect the output

`write_jsonl(morsel)` serializes a Draken Morsel to JSONL bytes directly in C++ —
no Python json module involved.  Here we write every row group of the full dataset
to a single file, then use `head` to confirm the output.

In [ ]:
from rugo.jsonl import write_jsonl

with open("space_missions.jsonl", "wb") as f:
    with parquet.read_parquet("space_missions.parquet") as reader:
        for morsel in reader:
            f.write(write_jsonl(morsel))

print(f"Written {os.path.getsize('space_missions.jsonl'):,} bytes to space_missions.jsonl")

In [ ]:
!head -n 5 space_missions.jsonl